# MobileNetV2 — aerial scene classification

Exploratory notebook for the team Keras flow. Prefer `train.py` for
comparable runs; use this notebook to inspect the model and iterate quickly.

```powershell
python train.py --model mobilenetv2 --run-name mobilenetv2_baseline
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

from shared import config
from shared.data import get_datasets
from shared.augment import get_augmentation
from shared.evaluate import evaluate

config.set_seed()
train_ds, val_ds = get_datasets()

from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


def build_model(
    num_classes: int, augmentation: keras.Sequential
) -> keras.Model:
    """Build and compile a frozen-backbone MobileNetV2 classifier."""
    inputs = keras.Input(
        shape=(config.IMAGE_SIZE, config.IMAGE_SIZE, 3), name="image"
    )

    # 1. Shared augmentation (raw [0, 255] in, raw [0, 255] out).
    x = augmentation(inputs)

    # 2. Backbone-specific preprocessing.
    x = preprocess_input(x)

    # 3. Pretrained backbone as a frozen feature extractor.
    backbone = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(config.IMAGE_SIZE, config.IMAGE_SIZE, 3),
        pooling="avg",
    )
    backbone.trainable = False
    x = backbone(x, training=False)

    # 4. New classification head.
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="mobilenet_v2")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
# Build the model, then train it
augmentation = get_augmentation()
model = build_model(config.NUM_CLASSES, augmentation)
model.summary()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,            # small while experimenting; raise later
)

In [ ]:
evaluate(model, val_ds, model_name="mobilenetv2_notebook")